# 02 — Preprocessing and the `TotalCharges` decision

- Preprocessing lives in `src/churn/preprocessing.py`; this notebook only *uses* it.
- The stratified train/test split is made once. **The test set is not used anywhere in this notebook.**
- The `TotalCharges` question is answered with cross-validation on the training set only.
- Models here are fixed, untuned baselines used purely as measuring instruments. Model selection is Phase 5.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

from churn.config import RANDOM_STATE
from churn.data import TARGET_COLUMN, load_dataset, split_features_target, split_train_test
from churn.preprocessing import EXCLUDED_FEATURES, build_pipeline, build_preprocessor, model_features

pd.set_option("display.precision", 4)

## 1. Stratified train/test split

In [2]:
df = load_dataset()
train_df, test_df = split_train_test(df)  # 80/20, stratified on Churn, fixed seed

X_train, y_train = split_features_target(train_df)
del test_df  # held out until final evaluation; not touched in this notebook

pd.DataFrame({
    "rows": [len(df), len(train_df)],
    "churn_rate": [df[TARGET_COLUMN].mean(), y_train.mean()],
}, index=["full dataset", "train (80%)"])

,rows,churn_rate
full dataset,7043,0.2654
train (80%),5634,0.2654


## 2. What the preprocessor produces

Fitted on the training set only here, to inspect the output. During cross-validation it is re-fitted inside each training fold, because it is part of the pipeline.

In [3]:
print("Excluded:", EXCLUDED_FEATURES)
print("Model inputs:", len(model_features()), "columns")

preprocessor = build_preprocessor().fit(X_train)
X_train_encoded = pd.DataFrame(preprocessor.transform(X_train),
                               columns=preprocessor.get_feature_names_out(), index=X_train.index)
print("Encoded output:", X_train_encoded.shape)
X_train_encoded.head(3).T

Excluded: {'PhoneService': "Fully determined by MultipleLines ('No phone service' level); Cramér's V = 1.0 in EDA."}
Model inputs: 18 columns
Encoded output: (5634, 39)


,3738,3151,4860
tenure,0.1024,-0.7117,-0.7932
MonthlyCharges,-0.5220,0.3375,-0.8090
TotalCharges,-0.2623,-0.5036,-0.7499
gender_Male,1.0000,1.0000,1.0000
SeniorCitizen_1,0.0000,0.0000,0.0000
Partner_Yes,0.0000,1.0000,1.0000
Dependents_Yes,0.0000,1.0000,1.0000
PaperlessBilling_Yes,0.0000,0.0000,0.0000
MultipleLines_No,0.0000,1.0000,0.0000
MultipleLines_No phone service,1.0000,0.0000,1.0000


In [4]:
X_train_encoded[["tenure", "MonthlyCharges", "TotalCharges"]].describe().loc[["mean", "std"]].round(3)

,tenure,MonthlyCharges,TotalCharges
mean,-0.0,-0.0,0.0
std,1.0,1.0,1.0


## 3. Should `TotalCharges` be a feature?

EDA showed `TotalCharges ≈ tenure × MonthlyCharges` (Spearman 0.89 with tenure). It may be redundant, or it may add signal (e.g. through price changes or the non-linear product).

**Method** (decided before looking at results):
- Training set only; repeated stratified 5-fold CV (5 folds × 3 repeats = 15 fits per configuration, same folds for every configuration).
- Two untuned baselines so the answer isn't specific to one model family: Logistic Regression (linear) and HistGradientBoosting (trees).
- Metrics: ROC-AUC (ranking), PR-AUC / average precision (focus on the churn class), log loss (probability quality; lower is better).
- Compare **paired** per-fold differences (with − without), which removes fold-to-fold variation shared by both configurations.

**Decision rule:** keep `TotalCharges` only if it improves ROC-AUC or PR-AUC for either model by more than 2 standard errors of the paired difference. Otherwise drop it: fewer inputs, less collinearity for Logistic Regression, and one less field for API users to supply. (Repeated-CV folds overlap, so the standard error is approximate; treat it as a noise guide, not a formal test.)

In [5]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "log_loss": "neg_log_loss"}

models = {
    "LogisticRegression": (lambda: LogisticRegression(max_iter=2000), True),
    "HistGradientBoosting": (lambda: HistGradientBoostingClassifier(random_state=RANDOM_STATE), False),
}

fold_scores = {}
for model_name, (make_model, scale) in models.items():
    for include in (True, False):
        pipeline = build_pipeline(make_model(), include_total_charges=include, scale_numeric=scale)
        result = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
        fold_scores[(model_name, include)] = pd.DataFrame({
            "roc_auc": result["test_roc_auc"],
            "pr_auc": result["test_pr_auc"],
            "log_loss": -result["test_log_loss"],
        })

summary = pd.concat(
    {(m, "with" if inc else "without"): s.agg(["mean", "std"]).T for (m, inc), s in fold_scores.items()}
).unstack()
summary

mean                      std                 
                             roc_auc  pr_auc log_loss roc_auc  pr_auc log_loss
HistGradientBoosting with     0.8366  0.6499   0.4352  0.0107  0.0204   0.0165
                     without  0.8355  0.6467   0.4357  0.0120  0.0227   0.0180
LogisticRegression   with     0.8463  0.6612   0.4169  0.0121  0.0176   0.0136
                     without  0.8450  0.6590   0.4176  0.0120  0.0179   0.0132

In [6]:
rows = []
for model_name in models:
    diff = fold_scores[(model_name, True)] - fold_scores[(model_name, False)]
    for metric in diff.columns:
        mean, se = diff[metric].mean(), diff[metric].std(ddof=1) / np.sqrt(len(diff))
        better_with = (diff[metric] < 0) if metric == "log_loss" else (diff[metric] > 0)
        rows.append({
            "model": model_name, "metric": metric,
            "mean_diff (with - without)": mean, "std_error": se,
            "diff / SE": mean / se if se > 0 else np.nan,
            "folds where 'with' is better": f"{int(better_with.sum())}/{len(diff)}",
        })
paired = pd.DataFrame(rows).set_index(["model", "metric"])
paired

mean_diff (with - without)  std_error  \
model                metric                                            
LogisticRegression   roc_auc                       0.0012     0.0003   
                     pr_auc                        0.0021     0.0005   
                     log_loss                     -0.0007     0.0004   
HistGradientBoosting roc_auc                       0.0010     0.0009   
                     pr_auc                        0.0032     0.0023   
                     log_loss                     -0.0006     0.0013   

                               diff / SE folds where 'with' is better  
model                metric                                            
LogisticRegression   roc_auc      4.8297                        14/15  
                     pr_auc       4.5522                        13/15  
                     log_loss    -1.7192                        13/15  
HistGradientBoosting roc_auc      1.1228                        10/15  
                     pr_auc       1.3795                         8/15  
                     log_loss    -0.4510                         9/15

For context: how Logistic Regression splits weight between the two correlated features (fitted on the full training set, standardized inputs).

In [7]:
coefs = {}
for include in (True, False):
    pipe = build_pipeline(LogisticRegression(max_iter=2000), include_total_charges=include).fit(X_train, y_train)
    names = pipe.named_steps["preprocess"].get_feature_names_out()
    coefs["with TotalCharges" if include else "without TotalCharges"] = (
        pd.Series(pipe.named_steps["model"].coef_[0], index=names)
    )
pd.DataFrame(coefs).loc[["tenure", "MonthlyCharges", "TotalCharges"]].round(3)

,with TotalCharges,without TotalCharges
tenure,-1.249,-0.801
MonthlyCharges,-0.633,-0.532
TotalCharges,0.525,NaN


## 4. Decision: keep `TotalCharges`

**Evidence (training-set CV, 15 paired folds):**
- **Logistic Regression:** ROC-AUC +0.0012 (≈4.8 SE, better in 14/15 folds), PR-AUC +0.0021 (≈4.6 SE, 13/15). Log loss also slightly lower.
- **HistGradientBoosting:** ROC-AUC +0.0010, PR-AUC +0.0032, but within noise (≈1.1–1.4 SE, 8–10/15 folds).

By the pre-set rule (improvement > 2 SE for either model), `TotalCharges` is **kept**.

**How to read this honestly:**
- The gain is *consistent* but *small* (about 0.001 ROC-AUC). It is not a meaningful business improvement on its own; it simply does not justify removing a feature that helps.
- A plausible reason it helps the linear model: `TotalCharges` ≈ `tenure × MonthlyCharges` is a non-linear combination that Logistic Regression cannot build from the two inputs by itself. Trees can approximate it, which fits the smaller, noisy gain there.
- **Collinearity consequence:** with `TotalCharges` included, Logistic Regression splits weight between correlated features (`tenure` −1.25, `TotalCharges` +0.53). The *positive* `TotalCharges` coefficient does not mean "higher total charges → more churn"; it is conditional on `tenure` and `MonthlyCharges`. Individual linear coefficients must not be read in isolation — relevant for explainability in Phase 8.

**Other decisions**
- `PhoneService` excluded (fully encoded by `MultipleLines`).
- No engineered features added. Candidates from EDA (support add-on count, tenure bands) are either linear combinations of existing one-hot columns or approximations tree models learn directly; they would be added only if Phase 5–7 validation shows a gap they could fill.